# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [1]:
import pandas as pd
corpus = pd.read_csv("data/wikipedia_text_corpus.csv")

In [8]:
print(corpus.columns)

Index(['Unnamed: 0', 'text', 'tokens'], dtype='str')


In [ ]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

# Descargar stopwords
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

# Stemmer
stemmer = PorterStemmer()

# Función de preprocesamiento
def preprocess_text(text):
    # Tokenización
    tokens = re.findall(r'\b[a-z]+\b', str(text).lower())

    # Eliminar stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    return tokens

# Aplicar al DataFrame
corpus["tokens"] = corpus["text"].apply(preprocess_text)

# Mostrar teto
print(corpus[["text", "tokens"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                                text  \
0  Anovo\n\nAnovo (formerly A Novo) is a computer...   
1  Battery indicator\n\nA battery indicator (also...   
2  Bob Pease\n\nRobert Allen Pease (August 22, 19...   
3  CAVNET\n\nCAVNET was a secure military forum w...   
4  CLidar\n\nThe CLidar is a scientific instrumen...   

                                              tokens  
0  [anovo, anovo, formerli, novo, comput, servic,...  
1  [batteri, indic, batteri, indic, also, known, ...  
2  [bob, peas, robert, allen, peas, august, june,...  
3  [cavnet, cavnet, secur, militari, forum, becam...  
4  [clidar, clidar, scientif, instrument, use, me...  


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = corpus["text"].astype(str)

# =========================
# Vectorización TF-IDF
# =========================
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(documents)

print("Shape matriz de TF-IDF:")
print(tfidf_matrix.shape)

# Queries de prueba
queries = [
    "city government",
    "Digital Assets",
    "history of europe",
    "climate change effects",
    "space exploration nasa",
    "world war",
    "human brain function",
    "internet and technology",
    "renewable energy",
    "web hosting"
]


def retrieve_documents(query, top_k=3):

    # Vectorizar query
    query_vector = vectorizer.transform([query])

    # Similitud coseno
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # indices más relevantes
    top_indices = similarities.argsort()[-top_k:][::-1]

    print(f"\nQUERY: {query}\n")

    for rank, idx in enumerate(top_indices, start=1):
        print(f"Top {rank} | Score: {similarities[idx]:.4f}")
        print(documents.iloc[idx][:300])  # Mostrar primeros caracteres
        print("-" * 80)


# Evaluar las 10 queries
for query in queries:
    retrieve_documents(query)

Shape matriz de TF-IDF:
(10859, 148411)

QUERY: city government

Top 1 | Score: 0.3114
I-City

i-City is a 72 acre ICT-based urban development beside the Federal Highway in Section 7, Shah Alam, Selangor, Malaysia. Planned by architect Jon A. Jerde, i-City was designed as a fully integrated intelligent city, comprising corporate, leisure and residential components such as a 1 million 
--------------------------------------------------------------------------------
Top 2 | Score: 0.2490
3D city models

3D city models are digital models of urban areas that represent terrain surfaces, sites, buildings, vegetation, infrastructure and landscape elements in three-dimensional scale as well as related objects (e.g., city furniture) belonging to urban areas. Their components are described 
--------------------------------------------------------------------------------
Top 3 | Score: 0.2329
HITEC City

The Hyderabad Information Technology and Engineering Consultancy City, abbreviated as HITEC C

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [16]:
from rank_bm25 import BM25Okapi

for query in queries:
    bm25 = BM25Okapi(corpus)

In [17]:
import numpy as np

for query in queries:
    scores = bm25.get_scores(query)

    top_idx = np.argsort(scores)[::-1]  # orden descendente

    print("\nQUERY:", query)
    for i in top_idx[:3]:
        print(f"Doc {i} | Score: {scores[i]:.4f}")


QUERY: city government
Doc 0 | Score: 1.0279
Doc 2 | Score: 0.8324
Doc 1 | Score: 0.2707

QUERY: Digital Assets
Doc 2 | Score: 1.7534
Doc 0 | Score: 0.8727
Doc 1 | Score: 0.2130

QUERY: history of europe
Doc 2 | Score: 2.2883
Doc 0 | Score: 0.9113
Doc 1 | Score: 0.1931

QUERY: climate change effects
Doc 0 | Score: 2.2979
Doc 2 | Score: 0.8820
Doc 1 | Score: 0.3862

QUERY: space exploration nasa
Doc 0 | Score: 2.6959
Doc 2 | Score: 2.3875
Doc 1 | Score: 0.8161

QUERY: world war
Doc 0 | Score: 1.2510
Doc 2 | Score: 0.5349
Doc 1 | Score: 0.0000

QUERY: human brain function
Doc 0 | Score: 2.3181
Doc 2 | Score: 0.7828
Doc 1 | Score: 0.0776

QUERY: internet and technology
Doc 0 | Score: 2.0171
Doc 2 | Score: 1.5656
Doc 1 | Score: 0.4061

QUERY: renewable energy
Doc 0 | Score: 1.1438
Doc 2 | Score: 0.3471
Doc 1 | Score: 0.2887

QUERY: web hosting
Doc 2 | Score: 1.2185
Doc 0 | Score: 0.5139
Doc 1 | Score: 0.1354


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 